In [ ]:
## !pip install transformers sentence-transformers faiss-cpu gradio pymupdf

In [ ]:
## !pip install -U transformers

In [ ]:
## !pip install groq

In [ ]:
from groq import Groq

client = Groq(api_key="Your API Key")   #Creates Groq client to call LLM API

In [ ]:
import fitz  ##read PDFs
import faiss     ## vector Search
import numpy as np   ## arrays
import gradio as gr   ## UI
from sentence_transformers import SentenceTransformer ## Used for converting text → embeddings (vectors)
from transformers import pipeline

In [ ]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
 embed_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')  #Loads embedding model
                                                                              #Converts text into numerical vectors for similarity search

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
chunks = []   ##all text pieces
index = None   ## FAISS index
chat_history = []    ## for conversation (not used fully yet)

In [ ]:
def load_pdf(file):
    doc = fitz.open(file)  ## Opens PDF file
    #Extracts text from every page
    text = ""
    for page in doc:
        text += page.get_text()
    #Returns full PDF text
    return text

In [ ]:
def chunk_text(text, chunk_size=700, overlap=150):       ##Breaks text into smaller pieces and Overlap ensures context continuity
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):       ## Sliding window chunking
        chunks.append(text[i:i+chunk_size])
    return chunks

In [ ]:
def create_index(chunks):
    embeddings = embed_model.encode(chunks)      ##Converts each chunk → vector
    dim = embeddings.shape[1]             ## Gets vector dimension

    index = faiss.IndexFlatL2(dim)     ##Creates FAISS index (L2 distance search)
    index.add(np.array(embeddings))    ## Stores embeddings in FAISS

    return index

In [ ]:
def search(query, index, chunks, k=12):
    query_vec = embed_model.encode([query])   # Converts query → embedding
    distances, indices = index.search(np.array(query_vec), k)    ##Finds top-k similar chunks

    results = [chunks[i] for i in indices[0]]    ## Gets actual text chunks

    return results, distances

In [ ]:
import re

# -----------------------------
# Clean Retrieved Chunks
# -----------------------------
def clean_chunks(chunks):
    cleaned = []
    for chunk in chunks:
        # remove MCQ options (A. B. C. D.)
        chunk = re.sub(r'\b[A-D]\.\s.*', '', chunk)   ## Removes MCQ options (A, B, C, D)

        # remove extra newlines
        chunk = re.sub(r'\n+', '\n', chunk)   ## Removes extra newlines

        cleaned.append(chunk.strip())

    return cleaned

In [ ]:
def filter_chunks(chunks, query):              ## Keyword-based filtering (improves relevance)
    query_words = set(query.lower().split())   ## Break query into words

    scored = []
    for chunk in chunks:
        chunk_words = set(chunk.lower().split())
        score = len(query_words & chunk_words)     ## Measures overlap between query and chunk
        scored.append((score, chunk))

    scored.sort(reverse=True)     ## sorts best chunks first

    return [c for s, c in scored if s > 0][:8]

In [ ]:
def generate_answer(query, context):
    try:
      ## Creates prompt for LLM & Forces model to use only retrieved data
        prompt = f"""
Answer the question using ONLY the context.

Context:
{context}

Question: {query}

Answer:
"""

        response = client.chat.completions.create(       ## Calls Groq LLM
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],     ##Sends prompt as chat message
            temperature=0.2                             ## Low randomness → more accurate answers
        )

        return response.choices[0].message.content.strip()      ## Extracts final answer

    except Exception as e:
        import traceback
        traceback.print_exc()   # 🔥 FULL ERROR
        return f"❌ API ERROR: {str(e)}"

In [ ]:
def process_pdfs(files):
    global chunks, index, chat_history

    all_text = ""
    for file in files:
        all_text += load_pdf(file.name)     ## Reads all PDFs

    chunks = chunk_text(all_text)      ## Splits into chunks
    index = create_index(chunks)      ## Builds FAISS index

    chat_history = []

    return f"✅ Processed {len(files)} PDF(s) | {len(chunks)} chunks created"

In [ ]:
def ask_question(query):
    global index, chunks

    if index is None:
        return "⚠️ Please upload and process documents first."

    # 🔹 Step 1: Retrieve chunks
    relevant_chunks, _ = search(query, index, chunks)    ## Retrieve similar chunks

    # 🔹 Step 2: Clean chunks
    relevant_chunks = clean_chunks(relevant_chunks)     ## Clean text

    # 🔹 Step 3: Filter chunks (IMPORTANT)
    relevant_chunks = filter_chunks(relevant_chunks, query)    ## Improve relevance

    # 🔹 Step 4: Debug (check what's being sent)
    print("\n🔍 TOP CHUNKS:\n")
    for c in relevant_chunks[:5]:
        print("-", c[:200])

    # 🔹 Step 5: Build context
    context = "\n".join(relevant_chunks[:6])      ## Combine top chunks

    print("\n📦 FINAL CONTEXT:\n", context[:1000])  # debug

    # 🔹 Step 6: Generate answer
    answer = generate_answer(query, context)    ## Send to LLM

    # 🔹 Step 7: Confidence check (relaxed)
    if len(answer.strip()) < 20:                ## Basic validation
        return "⚠️ Weak answer. Try rephrasing.\n\n" + answer

    return f"💡 Answer:\n{answer}"

In [ ]:
with gr.Blocks() as demo:      ## Creates UI
    gr.Markdown("# 🤖 Smart Document Assistant (Advanced RAG)")

    file_input = gr.File(file_count="multiple", label="Upload PDF(s)")    # Upload PDFs
    process_btn = gr.Button("Process Documents")
    status = gr.Textbox(label="Status")

    chatbot = gr.Chatbot()            ## Chat UI
    msg = gr.Textbox(label="Ask something")

    def respond(message, chat_history_ui):      ## Handles user input
        reply = ask_question(message)
        chat_history_ui.append((message, reply))
        return "", chat_history_ui

    process_btn.click(process_pdfs, inputs=file_input, outputs=status)    ## Button action
    msg.submit(respond, [msg, chatbot], [msg, chatbot])    ## Send message

demo.launch()

/tmp/ipykernel_20475/1518785693.py:8: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_20475/1518785693.py:8: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a987f77b2362a9b3fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
